# 04 — Train Unet2D adv-NO on magic-window export

Trains adv-NO on **2013–2022** data exported by [01_export_magic_window.ipynb](01_export_magic_window.ipynb) for rollout eval in [05_evaluate_adv_no_magic_window_rollout.ipynb](05_evaluate_adv_no_magic_window_rollout.ipynb).

**Change only `EXPORT_KEY`** in the setup cell below (same model/settings for all datasets). Run name and output folder are derived automatically.

| Item | Value |
|---|---|
| Data root | `Magic_window/data/{EXPORT_KEY}/` |
| Train export | `sample_13_22_train_uvbathy_dt360` (u, v, bathymetry) |
| Val export | `sample_13_22_val_uvbathy_dt360` (merged at train time when using chronological split) |
| Test export (nb 05) | `sample_23_uvbathy_dt360` |
| Run dir | `Models/adv_no/adv_no_uv_13_22_{EXPORT_KEY}_dt360_vgg_warm10_adv60_wd1e-3` |

**Train/val split (default):** merge train+val exports, then **random 70% train / 30% val** (`VAL_SPLIT = 0.3`, `VAL_SPLIT_MODE = "random"`, `VAL_SPLIT_SEED = 42`). **Validation metric:** masked **L1** on u,v. **Optimizer:** `AdamW`, `lr=2e-4`, `weight_decay=1e-3`, **100 epochs**.

**Run order:** setup → export check → dataset stats → train config → preview → train → loss curves.

**Saved for comparison:** `{RUN_DIR}/dataset_stats.json`, `{RUN_DIR}/best_model.pth`, `norm_stats.pt`, plus `Models/adv_no/comparison_index.json`


In [1]:
import json
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MW_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Magic_window" else NOTEBOOK_DIR / "Magic_window"
ROOT = MW_DIR.parent
if str(MW_DIR) not in sys.path:
    sys.path.insert(0, str(MW_DIR))

import importlib
import run_registry as _run_registry
importlib.reload(_run_registry)

from window_paths import train_prefix, val_prefix, test_prefix, split_manifest_path, window_data_dir
from run_registry import (
    RUN_TAG,
    collect_dataset_stats,
    run_dir,
    run_name,
    weight_decay_run_tag,
    save_dataset_stats,
    update_comparison_index,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = MW_DIR / "data"
MODELS_DIR = MW_DIR / "Models" / "adv_no"
LOG_DIR = MW_DIR / "logs"
PNG_DIR = MW_DIR / "PNG"
for _d in (DATA_DIR, MODELS_DIR, LOG_DIR, PNG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

import src.train.fix_window_train as _fwt
import src.train.fix_window_viz as _fwv
import src.train.losses as _losses
import src.train.adv_no_train as _adv
importlib.reload(_losses)
importlib.reload(_fwt)
importlib.reload(_fwv)
importlib.reload(_adv)

from src.train.adv_no_train import AdvNOTrainConfig, run_adv_no_training
from src.train.fix_window_train import (
    load_merged_training_arrays,
    load_training_arrays,
    resolve_train_val_indices,
    prepare_train_val_tensors,
)


def load_notebook_training_arrays(cfg):
    """Load arrays for preview/config cells; mirrors run_adv_no_training."""
    if cfg.merge_val_export_prefix and not cfg.val_data_prefix:
        return load_merged_training_arrays(cfg), None
    data, mask, day, hour, crop_h, crop_w = load_training_arrays(cfg)
    if cfg.val_data_prefix:
        val_tensors = load_training_arrays(cfg, data_prefix=cfg.val_data_prefix)
        return (data, mask, day, hour, crop_h, crop_w), val_tensors
    return (data, mask, day, hour, crop_h, crop_w), None

# --- ONLY CHANGE THIS when switching datasets ---
EXPORT_KEY = "magicwin_bathy_northeast"
# EXPORT_KEY = "legacy_smallfixwin_45deg"
# EXPORT_KEY = "magicwin_reference"

# Export component tag (must match 01_export_magic_window.ipynb EXPORT_COMPONENTS)
EXPORT_COMPONENT = "uvbathy"  # u + v + static bathymetry in sims (3 channels)
# EXPORT_COMPONENT = "uv"  # u+v only (old)

DT = 360
DATA_PREFIX = train_prefix(DATA_DIR, EXPORT_KEY, component=EXPORT_COMPONENT, dt=DT)
VAL_DATA_PREFIX = val_prefix(DATA_DIR, EXPORT_KEY, component=EXPORT_COMPONENT, dt=DT)
TEST_DATA_PREFIX = test_prefix(DATA_DIR, EXPORT_KEY, component=EXPORT_COMPONENT, dt=DT)
# Old uv-only paths:
# DATA_PREFIX = train_prefix(DATA_DIR, EXPORT_KEY, component="uv", dt=DT)
# VAL_DATA_PREFIX = val_prefix(DATA_DIR, EXPORT_KEY, component="uv", dt=DT)
# TEST_DATA_PREFIX = test_prefix(DATA_DIR, EXPORT_KEY, component="uv", dt=DT)
MAX_SAMPLES = None  # use all exported sequences

EPOCHS = 100
OPTIMIZER = "adamw"
WEIGHT_DECAY = 1e-4

BASE_RUN_TAG = RUN_TAG
RUN_TAG = weight_decay_run_tag(BASE_RUN_TAG, WEIGHT_DECAY)
RUN_NAME = run_name(EXPORT_KEY, dt=DT, run_tag=RUN_TAG)
RUN_DIR = run_dir(MODELS_DIR, EXPORT_KEY, dt=DT, run_tag=RUN_TAG)

# --- loss spatial / coastal options (disabled) ---
# Coastal buffer: dilates land mask → less aggressive loss near coast. Do NOT use.
# LOSS_MASK_BUFFER_PX = 4
# Center-weighted loss: down-weights crop edges. Do NOT use for magic-window coastal.
# LOSS_SPATIAL_MODE = "gaussian_center"
# LOSS_GAUSSIAN_SIGMA = 32.0
LOSS_SPATIAL_MODE = "full"  # uniform L1/perceptual/adv over all ocean pixels (mask==0)

LOSS_MODE = "adv_no"
LOSS_METRIC = "components"
LOSS_GAUSSIAN_SIGMA = 32.0  # unused unless LOSS_SPATIAL_MODE == "gaussian_center"
CHANNEL_NAMES = ("u", "v", "bathymetry")
iD = 3  # model in/out channels; bathymetry increment ~0 (static field)
# CHANNEL_NAMES = ("u", "v")
# iD = 2
BETA = 0.1
LAMBDA_PERC = 0.1
PERCEPTUAL_BACKEND = "vgg"
PERCEPTUAL_WEIGHTS = (0.1, 0.1, 1.0, 1.0, 1.0)
LPIPS_NET = "alex"
PERCEPTUAL_WARMUP_EPOCHS = 10
WARMUP_EPOCHS = 30
D_LR_RATIO = 0.5
D_SKIP_THRESHOLD = 0.2
USE_COSINE_LR = True
VAL_SPLIT = 0.3  # 30% validation, 70% train
VAL_SPLIT_MODE = "random"  # random 70/30 split (seed below)
MERGE_VAL_EXPORT = True  # merge train+val .npy exports before split
VAL_RECONSTRUCTION_METRIC = "l1"  # best checkpoint + val curves use masked uv L1
# VAL_SPLIT_MODE = "chronological"  # first 70% train, last 30% val by (day, hour)
# VAL_RECONSTRUCTION_METRIC = "rmse"
# VAL_SPLIT_MODE = "export"  # use random export-time split from 01_export_magic_window.ipynb
# MERGE_VAL_EXPORT = False
VAL_SPLIT_SEED = 42
DETACHED = False
SHOW_INLINE_PLOTS = False
PREVIEW_LOSS_MASK = True
PREVIEW_VAL_MASK = True
PREVIEW_VAL_MASK_SAMPLES = 3
PROTECT_EXISTING_RUN = False
FILTER_FULLY_OCEAN_SAMPLES = False
DEVICE = "cuda:6"
VIZ_EVERY = 1
VIZ_SAMPLES = 1
VIZ_SEED = 42
VIZ_RESIDUAL_METRIC = "l1"

print("EXPORT_COMPONENT:", EXPORT_COMPONENT)
print("EXPORT_KEY:", EXPORT_KEY)
print("DATA_PREFIX:", DATA_PREFIX)
print("VAL_DATA_PREFIX:", VAL_DATA_PREFIX)
print("TEST_DATA_PREFIX:", TEST_DATA_PREFIX)
print("RUN_NAME:", RUN_NAME)
print("RUN_DIR:", RUN_DIR)
print("epochs:", EPOCHS)
print(f"Optimizer: {OPTIMIZER}, lr=2e-4, weight_decay={WEIGHT_DECAY}, epochs={EPOCHS}")
print("val split:", f"{int((1 - VAL_SPLIT) * 100)}% train / {int(VAL_SPLIT * 100)}% val", f"({VAL_SPLIT_MODE}, seed={VAL_SPLIT_SEED})")
print("val metric:", VAL_RECONSTRUCTION_METRIC)
print("max_samples:", MAX_SAMPLES, "(all export sequences)")
print("rollout eval:", MW_DIR / "05_evaluate_adv_no_magic_window_rollout.ipynb")

EXPORT_COMPONENT: uvbathy
EXPORT_KEY: magicwin_bathy_northeast
DATA_PREFIX: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360
VAL_DATA_PREFIX: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360
TEST_DATA_PREFIX: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_23_uvbathy_dt360
RUN_NAME: adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4
RUN_DIR: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4
epochs: 100
Optimizer: adamw, lr=2e-4, weight_decay=0.0001, epochs=100
val split: 70% train / 30% val (random, seed=42)
val metric: l1
max_samples: None (all export sequences)
rollout eval: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/05_evaluate_adv_no_magic_window_rollout.ipynb


In [2]:
assert "DATA_PREFIX" in globals(), "Run the setup cell above first."
print("Ready:", DATA_PREFIX)
print("Val export:", VAL_DATA_PREFIX)
print("Next: export check → train config → preview → train")


Ready: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360
Val export: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360
Next: export check → train config → preview → train


In [3]:
import numpy as np

if "DATA_PREFIX" not in globals():
    raise RuntimeError("Run the setup cell above first.")

for label, prefix in (("train", DATA_PREFIX), ("val", VAL_DATA_PREFIX)):
    sims_path = Path(f"{prefix}_sims.npy")
    if not sims_path.exists():
        raise FileNotFoundError(
            f"Missing {label} export: {sims_path}\n"
            "Run Magic_window/01_export_magic_window.ipynb (train + val exports)."
        )

train_sims = np.load(f"{DATA_PREFIX}_sims.npy", mmap_mode="r")
val_sims = np.load(f"{VAL_DATA_PREFIX}_sims.npy", mmap_mode="r")
mask = np.load(f"{DATA_PREFIX}_mask.npy", mmap_mode="r")
window = np.load(f"{DATA_PREFIX}_window.npy")
land_frac = float(mask.mean())
static_mask = len(mask) <= 1 or np.all(mask == mask[0:1])
n_train_use = len(train_sims) if MAX_SAMPLES is None else min(int(MAX_SAMPLES), len(train_sims))

print(f"Train export: {DATA_PREFIX}")
print(f"  sims shape: {train_sims.shape}  (N, T, C, H, W)")
print(f"  sequences for training: {n_train_use} / {len(train_sims)}")
print(f"Val export: {VAL_DATA_PREFIX}")
print(f"  sims shape: {val_sims.shape}")
print(f"  mask: {land_frac:.2%} land | static={static_mask}")
print(f"  window [top, left, h, w, angle]: {window}")
if MERGE_VAL_EXPORT and VAL_SPLIT_MODE != "export":
    n_merged = len(train_sims) + len(val_sims)
    n_train_split = int(n_merged * (1.0 - VAL_SPLIT))
    n_val_split = n_merged - n_train_split
    print(
        f"  training split: merge {len(train_sims)}+{len(val_sims)}={n_merged} → "
        f"{n_train_split} train + {n_val_split} val "
        f"({VAL_SPLIT_MODE}, {VAL_SPLIT:.0%} val, seed={VAL_SPLIT_SEED})"
    )
else:
    print(
        f"  export-time split: {n_train_use} train + {len(val_sims)} val "
        f"(see {EXPORT_KEY}/sample_13_22_split.json)"
    )


Train export: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360
  sims shape: (2880, 4, 3, 128, 128)  (N, T, C, H, W)
  sequences for training: 2880 / 2880
Val export: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360
  sims shape: (720, 4, 3, 128, 128)
  mask: 0.00% land | static=True
  window [top, left, h, w, angle]: [100. 350. 128. 128.   0.]
  training split: merge 2880+720=3600 → 2520 train + 1080 val (random, 30% val, seed=42)


In [4]:
# Collect and save dataset statistics for cross-dataset comparison
assert "EXPORT_KEY" in globals(), "Run the setup cell first."

RUN_DIR.mkdir(parents=True, exist_ok=True)
dataset_stats = collect_dataset_stats(DATA_DIR, EXPORT_KEY, dt=DT, component=EXPORT_COMPONENT)
stats_path = save_dataset_stats(RUN_DIR, dataset_stats)
index_path = update_comparison_index(
    MODELS_DIR,
    run_name_value=RUN_NAME,
    export_key=EXPORT_KEY,
    run_dir_path=RUN_DIR,
    dataset_stats=dataset_stats,
)

print(f"Saved dataset stats: {stats_path}")
print(
    f"  train {dataset_stats['train']['n_sequences']} | "
    f"val {dataset_stats['val']['n_sequences']} | "
    f"test {dataset_stats['test']['n_sequences']} | "
    f"land {dataset_stats['train']['land_fraction']:.2%}"
)
print(f"Updated comparison index: {index_path}")


Saved dataset stats: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4/dataset_stats.json
  train 2880 | val 720 | test 319 | land 0.00%
Updated comparison index: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/comparison_index.json


In [5]:
train_config = AdvNOTrainConfig(
    data_prefix=DATA_PREFIX,
    run_name=RUN_NAME,
    dt_hours=DT,
    epochs=EPOCHS,
    batch_size=4,
    lr=2e-4,
    optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    d_lr=1e-4,
    d_lr_follow_g=True,
    d_lr_ratio=D_LR_RATIO,
    d_skip_threshold=D_SKIP_THRESHOLD,
    use_cosine_lr=USE_COSINE_LR,
    iD=iD,
    device=DEVICE,
    loss_mode=LOSS_MODE,
    loss_metric=LOSS_METRIC,
    loss_spatial_mode=LOSS_SPATIAL_MODE,
    loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
    loss_mask_buffer_px=0,
    filter_fully_ocean_samples=FILTER_FULLY_OCEAN_SAMPLES,
    max_samples=MAX_SAMPLES,
    protect_existing_run=PROTECT_EXISTING_RUN,
    preview_loss_mask=PREVIEW_LOSS_MASK,
    preview_val_mask=PREVIEW_VAL_MASK,
    beta=BETA,
    lambda_perc=LAMBDA_PERC,
    perceptual_backend=PERCEPTUAL_BACKEND,
    lpips_net=LPIPS_NET,
    perceptual_weights=PERCEPTUAL_WEIGHTS,
    perceptual_warmup_epochs=PERCEPTUAL_WARMUP_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    checkpoint_subdir=str(MODELS_DIR),
    isolate_runs=True,
    channel_names=CHANNEL_NAMES,
    viz_every=VIZ_EVERY,
    viz_samples=VIZ_SAMPLES,
    viz_seed=VIZ_SEED,
    viz_residual_metric=VIZ_RESIDUAL_METRIC,
    show_inline_plots=SHOW_INLINE_PLOTS,
    val_split=VAL_SPLIT,
    val_split_mode=VAL_SPLIT_MODE,
    val_split_seed=VAL_SPLIT_SEED,
    val_reconstruction_metric=VAL_RECONSTRUCTION_METRIC,
    val_data_prefix=None if MERGE_VAL_EXPORT else VAL_DATA_PREFIX,
    merge_val_export_prefix=VAL_DATA_PREFIX if MERGE_VAL_EXPORT else None,
)

(_data, _mask, _day, _hour, _, _), _val_tensors = load_notebook_training_arrays(train_config)
_n_total = len(_data)
if train_config.val_data_prefix:
    _n_val = len(_val_tensors[0])
else:
    _n_val = None
_train_idx, _val_idx = resolve_train_val_indices(
    train_config, _n_total, n_val=_n_val
)

RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / f"train_config_{RUN_NAME}.json"
config_path.write_text(json.dumps(asdict(train_config), indent=2, default=str))
print(f"Run directory: {RUN_DIR}")
print(f"Wrote config:  {config_path}")
print(f"  data: {DATA_PREFIX} | iD={iD} | loss_metric={LOSS_METRIC}")
if train_config.merge_val_export_prefix and not train_config.val_data_prefix:
    split_note = (
        f"merged {VAL_SPLIT_MODE} split ({VAL_SPLIT:.0%} val, seed={VAL_SPLIT_SEED}): "
        f"{DATA_PREFIX} + {VAL_DATA_PREFIX}"
    )
elif train_config.val_data_prefix:
    split_note = f"export-time split (val from {VAL_DATA_PREFIX})"
else:
    split_note = f"{VAL_SPLIT_MODE} split on {DATA_PREFIX}"
print(
    f"  sequences: {_n_total} total | "
    f"train {len(_train_idx)} | val {len(_val_idx)} ({split_note})"
)
if LOSS_MODE == "l1":
    print("  loss mode: L1-only (no perceptual / discriminator)")
elif LOSS_MODE == "adv_no":
    perc_name = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"
    if PERCEPTUAL_WARMUP_EPOCHS == 0 and WARMUP_EPOCHS == 0:
        print(f"  loss mode: adv-NO — L1 + {perc_name} + adversarial from epoch 1")
    else:
        print(
            f"  loss mode: adv-NO — L1 through ep {PERCEPTUAL_WARMUP_EPOCHS}, "
            f"+{perc_name} from ep {PERCEPTUAL_WARMUP_EPOCHS + 1}, "
            f"+adv from ep {WARMUP_EPOCHS + 1}"
        )
else:
    print(f"  loss mode: {LOSS_MODE!r}")
if LOSS_SPATIAL_MODE == "full":
    spatial_note = "uniform over whole valid field"
elif LOSS_SPATIAL_MODE == "gaussian_center":
    spatial_note = f"gaussian_center (σ={LOSS_GAUSSIAN_SIGMA}) — state⊙G"
else:
    spatial_note = LOSS_SPATIAL_MODE
print(f"  loss spatial: {spatial_note}")
print(f"  rollout: best_model.pth + norm_stats.pt → notebook 05")


Static land mask on all 2880 sequences (0.00% land) — Guiana-style per-pixel masking
  merged train: 2880 sequences from /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360


Static land mask on all 720 sequences (0.00% land) — Guiana-style per-pixel masking
  merged val: 720 sequences from /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360


/opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/src/train/fix_window_train.py:328: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647378361/work/torch/csrc/utils/tensor_numpy.cpp:206.)
  data = torch.from_numpy(np.asarray(sims_np))


Merged train+val exports: 3600 sequences (/opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360 + /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360)
Run directory: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4
Wrote config:  /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4/train_config_adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4.json
  data: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360 | iD=3 | loss_metric=components
  sequences: 3600 total | train 2520 | val 1080 (merged random split (30% val, seed=42): /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Mag

In [6]:
import numpy as np
from src.train.adv_no_train import _run_has_artifacts
from src.train.fix_window_train import SimulationDataset
from src.train.fix_window_viz import (
    fixed_viz_indices,
    save_loss_mask_preview,
    save_validation_mask_preview,
)

assert "train_config" in globals(), "Run the train config cell above first."
_mask_preview = np.load(f"{DATA_PREFIX}_mask.npy", mmap_mode="r")
land_frac = float(_mask_preview.mean())

if PROTECT_EXISTING_RUN and _run_has_artifacts(RUN_DIR, RUN_NAME):
    raise FileExistsError(
        f"Run already exists: {RUN_DIR}\n"
        "Change RUN_TAG to a new name before preview / training."
    )
elif _run_has_artifacts(RUN_DIR, RUN_NAME):
    print(f"Note: {RUN_DIR} exists — preview/training will overwrite prior results.")

if PREVIEW_LOSS_MASK:
    preview_path = save_loss_mask_preview(
        DATA_PREFIX,
        RUN_DIR / "viz" / "loss_mask_preview.png",
        sample_index=0,
        loss_spatial_mode=LOSS_SPATIAL_MODE,
        loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
        loss_mask_buffer_px=0,
        channel_names=CHANNEL_NAMES,
        show=SHOW_INLINE_PLOTS,
    )
    print(f"Loss mask preview: {preview_path}")
else:
    print("Loss mask preview skipped (PREVIEW_LOSS_MASK=False)")

if PREVIEW_VAL_MASK:
    (_data, _mask, _day, _hour, _, _), _val_tensors = load_notebook_training_arrays(train_config)
    if train_config.val_data_prefix:
        _val_data, _val_mask, _val_day, _val_hour = _val_tensors[:4]
    else:
        _val_data = _val_mask = _val_day = _val_hour = None
    (
        _tr_data,
        _tr_mask,
        _tr_day,
        _tr_hour,
        _va_data,
        _va_mask,
        _va_day,
        _va_hour,
        _train_idx,
        _val_idx,
    ) = prepare_train_val_tensors(
        _data,
        _mask,
        _day,
        _hour,
        train_config,
        val_data=_val_data,
        val_mask=_val_mask,
        val_day=_val_day,
        val_hour=_val_hour,
    )
    _train_set = SimulationDataset(_tr_data, _tr_mask, _tr_day, _tr_hour, iD=iD)
    _val_set = SimulationDataset(
        _va_data, _va_mask, _va_day, _va_hour,
        mean=_train_set.mean, std=_train_set.std, iD=iD,
    )
    _viz_idx = fixed_viz_indices(PREVIEW_VAL_MASK_SAMPLES, len(_val_set), VIZ_SEED)
    _global_viz_idx = [int(_val_idx[i]) for i in _viz_idx]
    val_mask_path = save_validation_mask_preview(
        _val_set,
        RUN_DIR / "viz" / "val_mask_preview.png",
        sample_indices=_viz_idx,
        global_sample_indices=_global_viz_idx,
        channel_names=CHANNEL_NAMES,
        loss_mask_buffer_px=0,
        show=SHOW_INLINE_PLOTS,
    )
    print(f"Val mask preview: {val_mask_path}")
else:
    print("Val mask preview skipped (PREVIEW_VAL_MASK=False)")


Note: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4 exists — preview/training will overwrite prior results.


Loss mask preview: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4/viz/loss_mask_preview.png
Static land mask on all 2880 sequences (0.00% land) — Guiana-style per-pixel masking
  merged train: 2880 sequences from /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360
Static land mask on all 720 sequences (0.00% land) — Guiana-style per-pixel masking
  merged val: 720 sequences from /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360
Merged train+val exports: 3600 sequences (/opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360 + /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360)
Train/val split (random, seed=42): 2520

## Train

Foreground by default (`DETACHED = False`). Set `DETACHED = True` to launch `scripts/train_adv_no_fix_window.py` in the background.


In [7]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
log_path = LOG_DIR / f"train_{RUN_NAME}.log"

if DETACHED:
    cmd = [
        sys.executable,
        str(ROOT / "scripts/train_adv_no_fix_window.py"),
        "--run-name",
        RUN_NAME,
        "--config",
        str(config_path),
    ]
    with open(log_path, "w") as log_file:
        proc = subprocess.Popen(
            cmd,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            cwd=ROOT,
        )
    print(f"Started background adv-NO training PID {proc.pid}")
    print(f"Log: {log_path}")
    print(f"Monitor: tail -f {log_path}")
    print("Stop with: kill", proc.pid)
else:
    class _Tee:
        def __init__(self, *streams):
            self.streams = streams

        def write(self, data):
            for stream in self.streams:
                stream.write(data)
                stream.flush()

        def flush(self):
            for stream in self.streams:
                stream.flush()

    print(f"Foreground training — log: {log_path}")
    print("Stop anytime: Kernel → Interrupt (or ■ button)\n")

    _stdout = sys.stdout
    with open(log_path, "w") as log_file:
        sys.stdout = _Tee(_stdout, log_file)
        try:
            run_adv_no_training(train_config)
        except KeyboardInterrupt:
            print("\nNotebook interrupt received — training stopped.")
        finally:
            sys.stdout = _stdout

    print(f"\nLog saved: {log_path}")


Foreground training — log: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/logs/train_adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4.log
Stop anytime: Kernel → Interrupt (or ■ button)

Static land mask on all 2880 sequences (0.00% land) — Guiana-style per-pixel masking
  merged train: 2880 sequences from /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360
Static land mask on all 720 sequences (0.00% land) — Guiana-style per-pixel masking
  merged val: 720 sequences from /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360


Merged train+val exports: 3600 sequences (/opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360 + /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360)
Dataset: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_train_uvbathy_dt360 + /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/data/magicwin_bathy_northeast/sample_13_22_val_uvbathy_dt360 (merged, random split, seed=42)
Samples: 3600 | crop 128 x 128
Using device: cuda:6
Loss mode: adv_no
Channels: iD=3 (u, v, bathymetry)
Reconstruction metric: components (uv L1)
Validation metric: l1 (uv L1)
Optimizer: adamw (lr=0.0002)
adamw weight_decay: 0.0001
Loss spatial focus: full valid field
lambda_perc: 0.1 | beta: 0.1
perceptual backend: VGG-19
perceptual_weights (VGG layers): (0.1, 0.1, 1.0, 1.0, 1.0)
Loss schedule: L1 through epoch 10, + pe

## Loss curves (after epoch 1+)

In [8]:
import matplotlib.pyplot as plt
import torch

history_path = RUN_DIR / "history" / f"training_history_{RUN_NAME}.pt"
state_path = RUN_DIR / f"train_state_{RUN_NAME}.json"
plot_path = RUN_DIR / "viz" / f"training_loss_curve_{RUN_NAME}.png"

if not history_path.exists():
    msg = [f"No history yet: {history_path}"]
    if state_path.exists():
        state = json.loads(state_path.read_text())
        msg.append(
            f"Training status: {state.get('status')} | "
            f"last completed epoch: {state.get('last_completed_epoch', 0)}/{state.get('epochs_planned', '?')}"
        )
    if log_path.exists():
        tail = log_path.read_text().splitlines()[-5:]
        msg.append("Recent log lines:")
        msg.extend(f"  {line}" for line in tail)
    msg.append("Re-run this cell after epoch 1 finishes.")
    print("\n".join(msg))
else:
    hist = torch.load(history_path, map_location="cpu", weights_only=False)
    epochs = hist["epoch"]

    if not epochs:
        print("History file exists but is empty. Wait for epoch 1 to finish, then re-run.")
    else:
        recon_label = "speed L1" if LOSS_METRIC == "speed" else "uv L1"
        fig, ax = plt.subplots(1, 3, figsize=(14, 4))
        ax[0].plot(epochs, hist["train_l1"], label=f"Train {recon_label}")
        ax[0].plot(epochs, hist["val_l1"], label=f"Val {recon_label}")
        ax[0].set_title(f"Reconstruction ({recon_label})")
        ax[0].legend()
        ax[0].grid(True, alpha=0.3)

        use_adv_no = LOSS_MODE == "adv_no"
        perc_warmup = PERCEPTUAL_WARMUP_EPOCHS if use_adv_no else 0
        adv_warmup = WARMUP_EPOCHS if use_adv_no else 0
        perc_title = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"

        if use_adv_no and any(hist.get("train_perceptual", [])):
            ax[1].plot(epochs, hist["train_perceptual"], label="Train perc")
            ax[1].plot(epochs, hist["val_perceptual"], label="Val perc")
            if perc_warmup > 0:
                ax[1].axvline(perc_warmup + 0.5, color="#2ca02c", linestyle=":", alpha=0.7)
            ax[1].set_title(f"Perceptual ({perc_title})")
            ax[1].legend()
            ax[1].grid(True, alpha=0.3)
        else:
            ax[1].set_title("Perceptual (starts after warmup)")
            ax[1].axis("off")

        if use_adv_no and any(hist.get("train_adv", [])):
            ax[2].plot(epochs, hist["train_adv"], label="Train adv")
            ax[2].plot(epochs, hist["train_d"], label="Train D")
            if adv_warmup > perc_warmup:
                ax[2].axvline(adv_warmup + 0.5, color="#888888", linestyle=":", alpha=0.7)
            ax[2].set_title("Adversarial terms")
            ax[2].legend()
            ax[2].grid(True, alpha=0.3)
        else:
            ax[2].set_title("Adversarial (starts after warmup)")
            ax[2].axis("off")

        plt.tight_layout()
        plot_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        if SHOW_INLINE_PLOTS:
            plt.show()
        else:
            plt.close(fig)
        print(f"Best val {recon_label}: {hist['best_val_loss']:.6f} at epoch {hist['best_epoch']}")
        print(f"Saved plot: {plot_path}")
        update_comparison_index(
            MODELS_DIR,
            run_name_value=RUN_NAME,
            export_key=EXPORT_KEY,
            run_dir_path=RUN_DIR,
            dataset_stats=dataset_stats if "dataset_stats" in globals() else None,
            training_summary={
                "best_val_loss": float(hist["best_val_loss"]),
                "best_epoch": int(hist["best_epoch"]),
                "epochs_completed": int(epochs[-1]) if epochs else 0,
            },
        )

Best val uv L1: 0.147505 at epoch 71
Saved plot: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4/viz/training_loss_curve_adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4.png


## Next: rollout evaluation

After training completes, open [05_evaluate_adv_no_magic_window_rollout.ipynb](05_evaluate_adv_no_magic_window_rollout.ipynb).

It loads `{RUN_DIR}/best_model.pth` and saves rollout metrics to `Magic_window/Test&Rollout/{RUN_NAME}/rollout_results.json`. Set the same `EXPORT_KEY` in notebook 05.

In [9]:
print(f"Model dir: {RUN_DIR}")
print(f"  best_model: {RUN_DIR / 'best_model.pth'}")
print(f"  norm_stats: {RUN_DIR / 'norm_stats.pt'}")
print("Open 05_evaluate_adv_no_magic_window_rollout.ipynb when training is done.")

Model dir: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4
  best_model: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4/best_model.pth
  norm_stats: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/Models/adv_no/adv_no_uv_13_22_magicwin_bathy_northeast_dt360_vgg_warm10_adv60_wd1e-4/norm_stats.pt
Open 05_evaluate_adv_no_magic_window_rollout.ipynb when training is done.
